我们主要使用Python来构建这些基础的量化模型，写库还是会用到C++等语言

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import *

首先是第一个获取数据，我们可以使用一些公开数据源，另外也可以使用离线数据，或者就是抛出不可用

In [ ]:
import tushare as ts
import akshare as ak

引用完这些基础库我们应该做一些变量的基础设置，统一放在前面方便更改

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

TS_TOKEN = ' '
TS_CODE = ' '
START_DATE = ' '
END_DATE = ' '

In [ ]:
ts.set_token(TS_TOKEN)
pro = ts.pro_api()

df = pro.daily(ts_code=TS_CODE, start_date=START_DATE, end_date=END_DATE)

先不搞那些乱七八糟的获取数据，我们先集中于这些模型吧，第一个就是二叉树binomialtree model

In [ ]:
from numpy import zeros, maximum, power, sqrt, around
def binomial_option(S, K, r, sigma, t, steps, output=0):
    '''
    构建欧式看涨期权的二叉树
    ------
    S       int/float   标的现价
    K       int/float   行权价
    r       float       无风险利率
    sigma   float       波动率
    t       int/float   到期时间（年）
    steps   int         步数
    output  int         0=价格矩阵 1=到期损益 2=期权价值 3=Delta
    -------
    '''
    dt = t / steps
    u = 1 + sigma * sqrt(dt)
    v = 1 - sigma * sqrt(dt)
    p = 0.5 + r * sqrt(dt) / (2 * sigma)
    df = 1 / (1 + r * dt)    # 折现因子
    
    px = zeros((steps + 1, steps + 1))     # 标的价格
    cp = zeros((steps + 1, steps + 1))     # 到期内在价值
    V = zeros((steps + 1, steps + 1))      # 期权价值
    delta = zeros((steps + 1, steps + 1))  # Delta
    
    # 前向方程：逐节点算价格与内在价值
    for j in range(steps + 1):
        for i in range(j + 1):
            px[i, j] = S * power(u, j - i) * power(v, i)
            cp[i, j] = maximum(px[i, j] - K, 0)
    
    # 后向方程：从到期日贴现回今天
    for j in range(steps, -1, -1):
        for i in range(j + 1):
            if j == steps:
                V[i, j] = cp[i, j]  # 到期：就是内在价值
                delta[i, j] = 1.0 if px[i, j] >= K else 0.0
            else:
                V[i, j] = df * (p * V[i, j + 1] + (1 - p) * V[i + 1, j + 1])
                delta[i, j] = (V[i, j + 1] - V[i + 1, j + 1]) / (px[i, j + 1] - px[i + 1, j + 1])
    
    results = (around(px, 2), around(cp, 2), around(V, 2), around(delta, 4))
    
    return results[output]

In [11]:
# 标的价格矩阵px，第j列是第j步的所有可能价格，第i行表示跌了i次，右上三角以外都是0
px = binomial_option(100, 100, 0.05, 0.2, 1, 4, output=0)
px

array([[100.  , 110.  , 121.  , 133.1 , 146.41],
       [  0.  ,  90.  ,  99.  , 108.9 , 119.79],
       [  0.  ,   0.  ,  81.  ,  89.1 ,  98.01],
       [  0.  ,   0.  ,   0.  ,  72.9 ,  80.19],
       [  0.  ,   0.  ,   0.  ,   0.  ,  65.61]])

In [12]:
# 到期内在价格矩阵cp = max{S_T - K, 0}
cp = binomial_option(100, 100, 0.05, 0.2, 1, 4, output=1)
cp

array([[ 0.  , 10.  , 21.  , 33.1 , 46.41],
       [ 0.  ,  0.  ,  0.  ,  8.9 , 19.79],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ]])

In [13]:
# 期权价格矩阵opx，左上角就是当天期权价格
opx = binomial_option(100, 100, 0.05, 0.2, 1, 4, output=2)
opx

array([[10.29, 15.88, 23.83, 34.33, 46.41],
       [ 0.  ,  3.39,  6.11, 10.99, 19.79],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ]])

In [14]:
# 对冲比率Delta
dlt = binomial_option(100, 100, 0.05, 0.2, 1, 4, output=3)
dlt

array([[0.6241, 0.8053, 0.9645, 1.    , 1.    ],
       [0.    , 0.3393, 0.5553, 0.9086, 1.    ],
       [0.    , 0.    , 0.    , 0.    , 0.    ],
       [0.    , 0.    , 0.    , 0.    , 0.    ],
       [0.    , 0.    , 0.    , 0.    , 0.    ]])

第二个是BS方程

In [ ]:
def NormalCDF(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def black_scholes_option(S, K, r, sigma, t, kind, output=0):
    '''
    Black-Scholes欧式期权闭式解
    ------
    S       int/float   标的现价
    K       int/float   行权价
    r       float       无风险利率
    sigma   float       波动率
    t       int/float   到期时间（年）
    kind    str         'call'或'put'
    output  int         0=价格 1=Delta 2=Gamma 3=Vega 4=Theta 5=Rho
    ------
    '''
    if t <= 0:
        intrinsic = maximum(S - K, 0.0) if kind == 'call' else maximum(K - S, 0.0)
        delta_expiry = 1.0 if (kind == 'call' and S > K) else (-1.0 if(kind == 'put' and S < K) else 0.0)
        results = (intrinsic, delta_expiry, 0.0, 0.0, 0.0, 0.0)
        return results[output]
    
    d1 = (log(S / K) + (r + 0.5 * sigma ** 2) * t) / (sigma * sqrt(t))
    d2 = d1 - sigma * sqrt(t)
    df = exp(-r * t)
    pdf_d1 = exp(-0.5 * d1 * d1) / sqrt(2 * pi)
    
    if kind == 'call':
        price = S * NormalCDF(d1) - K * df * NormalCDF(d2)
        delta = NormalCDF(d1)
        rho = K * t * df * NormalCDF(d2)
        theta = (-S * pdf_d1 * sigma / (2 * sqrt(t)) - r * K * df * NormalCDF(d2))
    else:
        price = K * df * NormalCDF(-d2) - S * NormalCDF(-d1)
        delta = NormalCDF(d1) - 1.0
        rho   = -K * t * df * NormalCDF(-d2)
        theta = (-S * pdf_d1 * sigma / (2 * sqrt(t)) + r * K * df * NormalCDF(-d2))
    
    gamma = pdf_d1 / (S * sigma * sqrt(t))
    vega  = S * pdf_d1 * sqrt(t)

    results = (price, delta, gamma, vega, theta, rho)
    return results[output]

     